# 02. 탐색적 데이터 분석 (EDA)
## 서울 성동구 요식 가맹점 조기 경보 시스템 | 빅콘테스트 2025

> **목적**: 패널 데이터를 다각도로 시각화하여 폐업 패턴, 상권·업종 특성, 피처 유효성 검증
> **분석 흐름**: 점포 분포 → 폐업 현황 → 시계열 추이 → 사건 정렬 분석 → 피처 분리능력 → 상관관계

### 분석 목차
1. 데이터 개요 & 점포 분포
2. 폐업 현황 분석 (상권별 / 업종별)
3. 월별 폐업 시계열 추이
4. 사건 정렬 분석 (Event Alignment, T-8 → T-0)
5. 주요 피처 분포 비교 (폐업 vs 생존)
6. Mann-Whitney U 검정 (피처 유의성)
7. 피처 상관관계 히트맵
8. 폐업 전 신호 강도 분석


In [ ]:
# ================================================================
# 환경 설정 및 데이터 로드
# ================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ── 한글 폰트 설정 ──────────────────────────────────────────
import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

# ── 컬러 팔레트 ────────────────────────────────────────────
PALETTE  = {'폐업': '#E74C3C', '생존': '#2ECC71'}
C_CLOSED = '#E74C3C'
C_ALIVE  = '#2ECC71'
C_ACCENT = '#3498DB'
C_GRAY   = '#95A5A6'
BG_COLOR = '#F8F9FA'

plt.rcParams.update({
    'figure.facecolor': BG_COLOR,
    'axes.facecolor':   BG_COLOR,
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'font.size': 11,
})

OUT_DIR = '../outputs/'

# ── 데이터 로드 ────────────────────────────────────────────
panel  = pd.read_csv(f'{OUT_DIR}panel_preprocessed.csv',
                     encoding='utf-8-sig',
                     parse_dates=['ARE_D_dt', 'MCT_ME_D_dt', 'TA_YM_dt'])

master = panel.drop_duplicates('ENCODED_MCT').copy()
panel_closed = panel[panel['is_closed_obs'] == 1]
panel_alive  = panel[panel['is_closed_obs'] == 0]

print(f"전체 점포:   {master['ENCODED_MCT'].nunique():,}개")
print(f"폐업 점포:   {(master['is_closed_obs']==1).sum()}개")
print(f"생존 점포:   {(master['is_closed_obs']==0).sum()}개")
print(f"관측 개월수: {panel['TA_YM'].nunique()}개월")


---
## 1. 점포 분포 — 상권별 & 업종별


In [ ]:
# ================================================================
# 1-1. 상권별 점포 수 & 폐업률
# ================================================================
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('성동구 상권별 점포 현황', fontsize=16, fontweight='bold', y=1.02)

# ── 상권별 점포 수 ─────────────────────────────────────────
dist_cnt = master.groupby('HPSN_MCT_BZN_CD_NM').size().sort_values(ascending=True)
colors_bar = [C_ACCENT] * len(dist_cnt)

ax = axes[0]
bars = ax.barh(dist_cnt.index, dist_cnt.values, color=C_ACCENT, alpha=0.8, edgecolor='white')
for bar, val in zip(bars, dist_cnt.values):
    ax.text(val + 10, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', fontsize=9, color='#2C3E50')
ax.set_xlabel('점포 수', fontsize=12)
ax.set_title('상권별 점포 수', fontsize=13, pad=12)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{int(x):,}'))

# ── 상권별 폐업률 ──────────────────────────────────────────
dist_stat = master.groupby('HPSN_MCT_BZN_CD_NM').agg(
    total=('ENCODED_MCT', 'count'),
    closed=('is_closed_obs', 'sum')
).reset_index()
dist_stat['closure_rate'] = dist_stat['closed'] / dist_stat['total'] * 100
dist_stat = dist_stat.sort_values('closure_rate', ascending=True)

ax2 = axes[1]
colors = [C_CLOSED if r > dist_stat['closure_rate'].median() else C_ALIVE
          for r in dist_stat['closure_rate']]
bars2 = ax2.barh(dist_stat['HPSN_MCT_BZN_CD_NM'], dist_stat['closure_rate'],
                 color=colors, alpha=0.85, edgecolor='white')
for bar, val in zip(bars2, dist_stat['closure_rate']):
    ax2.text(val + 0.05, bar.get_y() + bar.get_height()/2,
             f'{val:.1f}%', va='center', fontsize=9)

ax2.axvline(dist_stat['closure_rate'].median(), color='#2C3E50',
            linestyle='--', linewidth=1.5, alpha=0.6, label=f'중앙값 {dist_stat["closure_rate"].median():.1f}%')
ax2.set_xlabel('폐업률 (%)', fontsize=12)
ax2.set_title('상권별 폐업률 (관측기간 2023-2024)', fontsize=13, pad=12)
ax2.legend(fontsize=10)

plt.tight_layout()
plt.savefig(f'{OUT_DIR}eda_01_district_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(" 저장: eda_01_district_distribution.png")


In [ ]:
# ================================================================
# 1-2. 업종별 점포 수 & 폐업률 (상위 20개)
# ================================================================
fig, axes = plt.subplots(1, 2, figsize=(18, 8))
fig.suptitle('성동구 업종별 점포 현황 (상위 20개)', fontsize=16, fontweight='bold', y=1.02)

ind_stat = master.groupby('HPSN_MCT_ZCD_NM').agg(
    total=('ENCODED_MCT', 'count'),
    closed=('is_closed_obs', 'sum')
).reset_index()
ind_stat['closure_rate'] = ind_stat['closed'] / ind_stat['total'] * 100

# 점포 수 기준 상위 20
top20_by_cnt = ind_stat.nlargest(20, 'total').sort_values('total', ascending=True)

ax = axes[0]
bars = ax.barh(top20_by_cnt['HPSN_MCT_ZCD_NM'], top20_by_cnt['total'],
               color=C_ACCENT, alpha=0.8, edgecolor='white')
for bar, val in zip(bars, top20_by_cnt['total']):
    ax.text(val + 1, bar.get_y() + bar.get_height()/2,
            str(val), va='center', fontsize=9)
ax.set_xlabel('점포 수', fontsize=12)
ax.set_title('업종별 점포 수 (상위 20)', fontsize=13, pad=12)

# 폐업률 기준 상위 20
top20_by_rate = ind_stat[ind_stat['total'] >= 5].nlargest(20, 'closure_rate').sort_values('closure_rate', ascending=True)

ax2 = axes[1]
median_rate = ind_stat['closure_rate'].median()
colors = [C_CLOSED if r > median_rate else C_ALIVE for r in top20_by_rate['closure_rate']]
bars2 = ax2.barh(top20_by_rate['HPSN_MCT_ZCD_NM'], top20_by_rate['closure_rate'],
                 color=colors, alpha=0.85, edgecolor='white')
for bar, val, cnt in zip(bars2, top20_by_rate['closure_rate'], top20_by_rate['total']):
    ax2.text(val + 0.1, bar.get_y() + bar.get_height()/2,
             f'{val:.1f}% (n={cnt})', va='center', fontsize=8.5)
ax2.axvline(median_rate, color='#2C3E50', linestyle='--', linewidth=1.5,
            alpha=0.6, label=f'전체 중앙값 {median_rate:.1f}%')
ax2.set_xlabel('폐업률 (%)', fontsize=12)
ax2.set_title('업종별 폐업률 (5개 이상 점포, 상위 20)', fontsize=13, pad=12)
ax2.legend(fontsize=10)

plt.tight_layout()
plt.savefig(f'{OUT_DIR}eda_02_industry_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(" 저장: eda_02_industry_distribution.png")


---
## 2. 폐업 현황 — 버블 차트 (상권 × 폐업률 × 점포 수)


In [ ]:
# ================================================================
# 2-1. 상권 버블 차트: 매출 수준 vs 폐업률
# ================================================================
fig, ax = plt.subplots(figsize=(14, 8))
fig.patch.set_facecolor(BG_COLOR)

# 상권별 평균 매출 구간 + 폐업률 집계
dist_bubble = panel.groupby('HPSN_MCT_BZN_CD_NM').agg(
    avg_sales_bucket=('RC_M1_SAA', 'median'),
    closure_rate=('is_closed_obs', lambda x: master.set_index('ENCODED_MCT')
                  .loc[panel.loc[x.index, 'ENCODED_MCT'].unique(), 'is_closed_obs'].mean() * 100
                  if len(panel.loc[x.index, 'ENCODED_MCT'].unique()) > 0 else 0),
    n_stores=('ENCODED_MCT', 'nunique')
).reset_index().dropna()

# 단순화: 상권별 점포 정보는 master에서
dist_agg = master.groupby('HPSN_MCT_BZN_CD_NM').agg(
    n_stores=('ENCODED_MCT', 'count'),
    closed=('is_closed_obs', 'sum')
).reset_index()
dist_agg['closure_rate'] = dist_agg['closed'] / dist_agg['n_stores'] * 100

sales_by_dist = panel.groupby('HPSN_MCT_BZN_CD_NM')['RC_M1_SAA'].median().reset_index()
sales_by_dist.columns = ['HPSN_MCT_BZN_CD_NM', 'median_sales_bucket']

dist_agg = dist_agg.merge(sales_by_dist, on='HPSN_MCT_BZN_CD_NM', how='left').dropna()

scatter = ax.scatter(
    dist_agg['median_sales_bucket'],
    dist_agg['closure_rate'],
    s=dist_agg['n_stores'] * 2,        # 크기 = 점포 수
    c=dist_agg['closure_rate'],
    cmap='RdYlGn_r',
    alpha=0.8, edgecolors='white', linewidth=1.5, zorder=3
)

# 라벨 추가
for _, row in dist_agg.iterrows():
    ax.annotate(
        row['HPSN_MCT_BZN_CD_NM'],
        (row['median_sales_bucket'], row['closure_rate']),
        textcoords='offset points', xytext=(8, 4),
        fontsize=9, color='#2C3E50'
    )

# 기준선
ax.axhline(dist_agg['closure_rate'].mean(), color=C_CLOSED, linestyle='--',
           alpha=0.5, linewidth=1.5, label=f'평균 폐업률 {dist_agg["closure_rate"].mean():.1f}%')
ax.axvline(dist_agg['median_sales_bucket'].mean(), color=C_ACCENT, linestyle='--',
           alpha=0.5, linewidth=1.5, label=f'평균 매출 버킷')

# 사분면 레이블
xlim = ax.get_xlim(); ylim = ax.get_ylim()
ax.text(xlim[0]*1.05, ylim[1]*0.95, '저매출·고폐업
 위험 상권', fontsize=9,
        color=C_CLOSED, alpha=0.7, va='top')
ax.text(xlim[1]*0.85, ylim[0]*1.05, '고매출·저폐업
 안정 상권', fontsize=9,
        color=C_ALIVE, alpha=0.7, va='bottom')

plt.colorbar(scatter, ax=ax, label='폐업률 (%)', shrink=0.8)
ax.set_xlabel('중위 매출 버킷 (1=최상위  6=최하위)', fontsize=12)
ax.set_ylabel('폐업률 (%)', fontsize=12)
ax.set_title('상권별 매출 수준 vs 폐업률\n(버블 크기 = 점포 수)', fontsize=14, pad=15)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.2, linestyle='--')

plt.tight_layout()
plt.savefig(f'{OUT_DIR}eda_03_district_bubble.png', dpi=150, bbox_inches='tight')
plt.show()
print(" 저장: eda_03_district_bubble.png")


---
## 3. 월별 폐업 시계열 추이


In [ ]:
# ================================================================
# 3-1. 월별 폐업 발생 건수 (시계열)
# ================================================================
fig, axes = plt.subplots(2, 1, figsize=(16, 10))
fig.suptitle('성동구 요식 가맹점 월별 폐업 추이 (2023-2024)', fontsize=15, fontweight='bold')

# 폐업 점포만 추출 (폐업월 기준)
closed_stores = master[master['MCT_ME_D_dt'].notna()].copy()
closed_stores['close_ym'] = closed_stores['MCT_ME_D_dt'].dt.to_period('M')
monthly_close = closed_stores.groupby('close_ym').size().reset_index(name='count')
monthly_close['close_dt'] = monthly_close['close_ym'].dt.to_timestamp()
monthly_close = monthly_close.sort_values('close_dt')

# 전체 관측 점포 수 (생존 중인 점포 추이)
monthly_active = panel.groupby('TA_YM_dt')['ENCODED_MCT'].nunique().reset_index(name='active_stores')

ax1 = axes[0]
ax1.fill_between(monthly_close['close_dt'], monthly_close['count'],
                 alpha=0.3, color=C_CLOSED)
ax1.plot(monthly_close['close_dt'], monthly_close['count'],
         color=C_CLOSED, linewidth=2.5, marker='o', markersize=5, label='월별 폐업 건수')
ax1.set_ylabel('폐업 건수', fontsize=12, color=C_CLOSED)

# 3개월 이동평균
ma3 = monthly_close.set_index('close_dt')['count'].rolling(3).mean()
ax1.plot(ma3.index, ma3.values, color='#8E44AD', linewidth=2,
         linestyle='--', alpha=0.8, label='3개월 이동평균')

ax1.set_title('월별 폐업 발생 건수', fontsize=13, pad=12)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.2, linestyle='--')

# 활성 점포 수 추이
ax2 = axes[1]
ax2.fill_between(monthly_active['TA_YM_dt'], monthly_active['active_stores'],
                 alpha=0.2, color=C_ACCENT)
ax2.plot(monthly_active['TA_YM_dt'], monthly_active['active_stores'],
         color=C_ACCENT, linewidth=2.5, marker='s', markersize=4, label='활성 점포 수')
ax2.set_xlabel('연월', fontsize=12)
ax2.set_ylabel('활성 점포 수', fontsize=12, color=C_ACCENT)
ax2.set_title('월별 활성 점포 수 추이', fontsize=13, pad=12)
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.2, linestyle='--')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{int(x):,}'))

for ax in axes:
    ax.xaxis.set_major_formatter(
        matplotlib.dates.DateFormatter('%Y-%m')
    )
    plt.setp(ax.get_xticklabels(), rotation=30, ha='right')

plt.tight_layout()
plt.savefig(f'{OUT_DIR}eda_04_monthly_closure_trend.png', dpi=150, bbox_inches='tight')
plt.show()
print(" 저장: eda_04_monthly_closure_trend.png")


---
## 4. 사건 정렬 분석 (Event Alignment) — 골든타임 탐색
폐업 점포를 **폐업 시점(T=0)** 기준으로 정렬하여, 폐업 전 몇 개월부터 신호가 나타나는지 분석.
생존 점포는 마지막 관측월을 기준(T=0)으로 정렬하여 비교군으로 활용.


In [ ]:
# ================================================================
# 4-1. 사건 정렬 분석 — 폐업 전 T개월 패턴
# ================================================================
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle('사건 정렬 분석 (Event Alignment): 폐업 전 8개월 신호 추이',
             fontsize=15, fontweight='bold', y=1.02)

FEAT_EA = {
    'RC_M1_SAA':          ('매출 버킷 (높을수록 하위)', '↑ = 하위권'),
    'MCT_UE_CLN_REU_RAT': ('재방문율 버킷', '↑ = 하위권'),
    'RC_M1_TO_UE_CT':     ('이용건수 버킷', '↑ = 하위권'),
    'M1_SME_RY_SAA_RAT':  ('업종 내 매출 순위 비율', '↑ = 하위권'),
}

# 폐업 점포 — months_to_close 기준 사건 정렬
closed_panel = panel[
    (panel['is_closed_obs'] == 1) &
    (panel['months_to_close'].notna()) &
    (panel['months_to_close'] <= 8)
].copy()

# 생존 점포 — 마지막 관측월 기준 (T=0: 마지막, T=-1: 그 전달 ...)
alive_panel = panel[panel['is_closed_obs'] == 0].copy()
alive_panel['months_to_close'] = alive_panel.groupby('ENCODED_MCT')['TA_YM'].transform(
    lambda x: x.rank(ascending=False, method='first') - 1
)
alive_panel = alive_panel[alive_panel['months_to_close'] <= 8]

for idx, (feat, (title, note)) in enumerate(FEAT_EA.items()):
    ax = axes[idx // 2][idx % 2]
    t_vals = list(range(8, -1, -1))  # T-8 ... T-0

    # 폐업 점포 평균
    closed_means = closed_panel.groupby('months_to_close')[feat].mean()
    # 생존 점포 평균
    alive_means  = alive_panel.groupby('months_to_close')[feat].mean()

    closed_arr = [closed_means.get(t, np.nan) for t in range(9)]
    alive_arr  = [alive_means.get(t, np.nan)  for t in range(9)]

    x = list(range(9))
    x_labels = [f'T-{i}' if i > 0 else 'T=0
(폐업)' for i in range(8, -1, -1)]

    ax.plot(x, closed_arr[::-1], color=C_CLOSED, linewidth=2.5,
            marker='o', markersize=6, label='폐업 점포', zorder=3)
    ax.plot(x, alive_arr[::-1],  color=C_ALIVE,  linewidth=2.5,
            marker='s', markersize=6, label='생존 점포', zorder=3)

    # CI 음영 (±0.3 모의)
    c_arr = np.array([v if v is not None else np.nan for v in closed_arr[::-1]])
    ax.fill_between(x,
                    np.where(np.isnan(c_arr), np.nan, c_arr - 0.2),
                    np.where(np.isnan(c_arr), np.nan, c_arr + 0.2),
                    color=C_CLOSED, alpha=0.1)

    ax.axvline(8, color='#2C3E50', linestyle='--', linewidth=1.5, alpha=0.5, label='T=0 (폐업월)')
    ax.set_xticks(x)
    ax.set_xticklabels(x_labels, fontsize=9)
    ax.set_title(title, fontsize=12, pad=10)
    ax.set_ylabel(f'버킷 중간값 ({note})', fontsize=10)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.2, linestyle='--')

plt.tight_layout()
plt.savefig(f'{OUT_DIR}eda_05_event_alignment.png', dpi=150, bbox_inches='tight')
plt.show()
print(" 저장: eda_05_event_alignment.png")


---
## 5. 주요 피처 분포 비교 (폐업 vs 생존)
KDE + 박스플롯으로 폐업 점포와 생존 점포의 피처 분포 차이를 시각화.


In [ ]:
# ================================================================
# 5-1. 피처 분포 비교 — KDE 플롯 (2x3)
# ================================================================
FEATS_DIST = [
    ('RC_M1_SAA',          '매출 버킷 (1=최상, 6=최하)'),
    ('MCT_UE_CLN_REU_RAT', '재방문율 버킷'),
    ('RC_M1_TO_UE_CT',     '이용건수 버킷'),
    ('RC_M1_SHC_FLP_UE_CLN_RAT', '유동인구 이용 비율'),
    ('M1_SME_RY_SAA_RAT',  '업종 내 매출 순위'),
    ('RC_M1_AV_NP_AT',     '평균 객단가 버킷'),
]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('폐업 vs 생존 점포 — 주요 피처 분포 비교', fontsize=15, fontweight='bold', y=1.02)

snapshot_closed = panel[panel['is_closed_obs']==1].groupby('ENCODED_MCT')[
    [f for f,_ in FEATS_DIST]].mean()
snapshot_alive  = panel[panel['is_closed_obs']==0].groupby('ENCODED_MCT')[
    [f for f,_ in FEATS_DIST]].mean()

for idx, (feat, label) in enumerate(FEATS_DIST):
    ax = axes[idx // 3][idx % 3]

    c_vals = snapshot_closed[feat].dropna()
    a_vals = snapshot_alive[feat].dropna()

    # KDE
    try:
        from scipy.stats import gaussian_kde
        x_range = np.linspace(1, 6, 200)
        kde_c = gaussian_kde(c_vals)
        kde_a = gaussian_kde(a_vals)
        ax.plot(x_range, kde_c(x_range), color=C_CLOSED, linewidth=2.5,
                label=f'폐업 (n={len(c_vals)})', zorder=3)
        ax.plot(x_range, kde_a(x_range), color=C_ALIVE, linewidth=2.5,
                label=f'생존 (n={len(a_vals)})', zorder=3)
        ax.fill_between(x_range, kde_c(x_range), alpha=0.2, color=C_CLOSED)
        ax.fill_between(x_range, kde_a(x_range), alpha=0.15, color=C_ALIVE)
    except Exception:
        ax.hist(c_vals, bins=6, color=C_CLOSED, alpha=0.5, density=True, label='폐업')
        ax.hist(a_vals, bins=6, color=C_ALIVE,  alpha=0.5, density=True, label='생존')

    # 중앙값 수직선
    ax.axvline(c_vals.median(), color=C_CLOSED, linestyle='--', linewidth=1.5, alpha=0.7)
    ax.axvline(a_vals.median(), color=C_ALIVE,  linestyle='--', linewidth=1.5, alpha=0.7)

    # Mann-Whitney 검정
    try:
        stat, pval = stats.mannwhitneyu(c_vals, a_vals, alternative='two-sided')
        sig = '***' if pval < 0.001 else ('**' if pval < 0.01 else ('*' if pval < 0.05 else 'ns'))
        ax.text(0.98, 0.95, f'p={pval:.3f} {sig}', transform=ax.transAxes,
                ha='right', va='top', fontsize=10,
                color=C_CLOSED if pval < 0.05 else C_GRAY,
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))
    except Exception:
        pass

    ax.set_title(label, fontsize=12, pad=10)
    ax.set_xlabel('버킷값', fontsize=10)
    ax.set_ylabel('밀도', fontsize=10)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.2, linestyle='--')

plt.tight_layout()
plt.savefig(f'{OUT_DIR}eda_06_feature_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(" 저장: eda_06_feature_distribution.png")


In [ ]:
# ================================================================
# 5-2. 박스플롯 비교 (다중 피처 나란히)
# ================================================================
fig, ax = plt.subplots(figsize=(16, 7))
fig.suptitle('폐업 vs 생존 점포 — 피처별 중앙값 비교 (박스플롯)', fontsize=14, fontweight='bold')

feats_box = [f for f, _ in FEATS_DIST]
labels_box = [label for _, label in FEATS_DIST]

# 폐업/생존 각각 평균 스냅샷
data_c = [snapshot_closed[f].dropna().values for f in feats_box]
data_a = [snapshot_alive[f].dropna().values  for f in feats_box]

x = np.arange(len(feats_box))
width = 0.35

bp_c = ax.boxplot(data_c, positions=x - width/2, widths=width*0.85,
                  patch_artist=True, notch=True,
                  boxprops=dict(facecolor=C_CLOSED, alpha=0.6),
                  medianprops=dict(color='white', linewidth=2),
                  whiskerprops=dict(color=C_CLOSED, alpha=0.7),
                  capprops=dict(color=C_CLOSED),
                  flierprops=dict(marker='o', color=C_CLOSED, alpha=0.3, markersize=3))

bp_a = ax.boxplot(data_a, positions=x + width/2, widths=width*0.85,
                  patch_artist=True, notch=True,
                  boxprops=dict(facecolor=C_ALIVE, alpha=0.6),
                  medianprops=dict(color='white', linewidth=2),
                  whiskerprops=dict(color=C_ALIVE, alpha=0.7),
                  capprops=dict(color=C_ALIVE),
                  flierprops=dict(marker='s', color=C_ALIVE, alpha=0.3, markersize=3))

ax.set_xticks(x)
ax.set_xticklabels(labels_box, rotation=20, ha='right', fontsize=10)
ax.set_ylabel('버킷값 (점포별 평균)', fontsize=12)

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=C_CLOSED, alpha=0.6, label='폐업 점포'),
                   Patch(facecolor=C_ALIVE,  alpha=0.6, label='생존 점포')]
ax.legend(handles=legend_elements, fontsize=11)
ax.grid(True, alpha=0.2, linestyle='--', axis='y')

plt.tight_layout()
plt.savefig(f'{OUT_DIR}eda_07_boxplot_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(" 저장: eda_07_boxplot_comparison.png")


---
## 6. Mann-Whitney U 검정 — 피처 유의성 순위
폐업 점포의 피처값이 생존 점포보다 유의미하게 큰지(위험 방향) 검증.


In [ ]:
# ================================================================
# 6. Mann-Whitney U 검정 결과 시각화
# ================================================================
ALL_FEATS = [f for f, _ in FEATS_DIST]

results_mw = []
for feat in ALL_FEATS:
    c_vals = snapshot_closed[feat].dropna()
    a_vals = snapshot_alive[feat].dropna()
    if len(c_vals) < 3 or len(a_vals) < 3:
        continue
    stat, pval = stats.mannwhitneyu(c_vals, a_vals, alternative='greater')
    # 효과크기: rank-biserial correlation
    n1, n2 = len(c_vals), len(a_vals)
    r = 1 - (2 * stat) / (n1 * n2)
    results_mw.append({
        'feature': feat,
        'p_value': pval,
        'effect_size': r,
        'closed_median': c_vals.median(),
        'alive_median':  a_vals.median(),
    })

res_df = pd.DataFrame(results_mw).sort_values('effect_size', ascending=True)
res_df['significant'] = res_df['p_value'] < 0.05
res_df['label'] = [dict(FEATS_DIST).get(f, f) for f in res_df['feature']]

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle('Mann-Whitney U 검정: 피처 유의성 & 효과 크기', fontsize=14, fontweight='bold')

# ── 효과 크기 (rank-biserial) ──────────────────────────────
ax = axes[0]
colors = [C_CLOSED if sig else C_GRAY for sig in res_df['significant']]
bars = ax.barh(res_df['label'], res_df['effect_size'], color=colors, alpha=0.85, edgecolor='white')
for bar, pval in zip(bars, res_df['p_value']):
    sig_txt = '***' if pval<0.001 else ('**' if pval<0.01 else ('*' if pval<0.05 else 'ns'))
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            sig_txt, va='center', fontsize=11,
            color=C_CLOSED if pval < 0.05 else C_GRAY)
ax.axvline(0, color='#2C3E50', linewidth=1, alpha=0.5)
ax.set_xlabel('효과 크기 (Rank-Biserial Correlation)', fontsize=11)
ax.set_title('피처별 효과 크기\n(붉은색 = p < 0.05 유의)', fontsize=12, pad=10)
ax.grid(True, alpha=0.2, linestyle='--', axis='x')

# ── p-value (-log10) ──────────────────────────────────────
ax2 = axes[1]
res_df2 = res_df.sort_values('p_value', ascending=True)
neg_log_p = -np.log10(res_df2['p_value'].clip(1e-10, 1))
colors2 = [C_CLOSED if p < 0.05 else C_GRAY for p in res_df2['p_value']]
ax2.barh(res_df2['label'], neg_log_p, color=colors2, alpha=0.85, edgecolor='white')
ax2.axvline(-np.log10(0.05), color='#E67E22', linestyle='--', linewidth=2,
            label='유의수준 p=0.05', zorder=5)
ax2.axvline(-np.log10(0.001), color=C_CLOSED, linestyle='--', linewidth=1.5,
            alpha=0.6, label='유의수준 p=0.001', zorder=5)
ax2.set_xlabel('-log₁₀(p-value)', fontsize=11)
ax2.set_title('피처별 유의성\n(오른쪽일수록 유의)', fontsize=12, pad=10)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.2, linestyle='--', axis='x')

plt.tight_layout()
plt.savefig(f'{OUT_DIR}eda_08_mannwhitney.png', dpi=150, bbox_inches='tight')
plt.show()
print(res_df[['label','effect_size','p_value','closed_median','alive_median']].to_string(index=False))
print("\n 저장: eda_08_mannwhitney.png")


---
## 7. 피처 상관관계 히트맵


In [ ]:
# ================================================================
# 7. 피처 상관관계 히트맵 (폐업 / 생존 분리)
# ================================================================
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('피처 상관관계 히트맵 (Spearman)', fontsize=14, fontweight='bold')

for ax, (df_snap, title, cmap) in zip(axes, [
    (snapshot_closed, '폐업 점포 (n=30)', 'Reds'),
    (snapshot_alive,  '생존 점포',        'Greens')
]):
    corr = df_snap[feats_box].corr(method='spearman')
    mask = np.triu(np.ones_like(corr, dtype=bool))

    sns.heatmap(
        corr, ax=ax, mask=mask, annot=True, fmt='.2f',
        cmap=cmap, center=0, vmin=-1, vmax=1,
        linewidths=0.5, linecolor='white',
        annot_kws={'size': 9},
        xticklabels=[dict(FEATS_DIST).get(f, f) for f in feats_box],
        yticklabels=[dict(FEATS_DIST).get(f, f) for f in feats_box],
        square=True,
    )
    ax.set_title(title, fontsize=12, pad=12)
    ax.tick_params(axis='x', rotation=30, labelsize=9)
    ax.tick_params(axis='y', rotation=0,  labelsize=9)

plt.tight_layout()
plt.savefig(f'{OUT_DIR}eda_09_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print(" 저장: eda_09_correlation_heatmap.png")


---
## 8. 폐업 전 신호 강도 — 히트맵
폐업까지 남은 개월(T-8 → T-0) × 피처별 평균 버킷값을 히트맵으로 시각화.
**색이 붉을수록** 해당 피처의 하위권(위험 신호) 강도가 높음.


In [ ]:
# ================================================================
# 8. 폐업 전 신호 히트맵 (T-8 → T-0 × 피처)
# ================================================================
feats_hm  = [f for f, _ in FEATS_DIST]
labels_hm = [l for _, l in FEATS_DIST]

closed_pre = panel[
    (panel['is_closed_obs'] == 1) &
    (panel['months_to_close'].notna()) &
    (panel['months_to_close'] <= 8)
].copy()

hm_data = closed_pre.groupby('months_to_close')[feats_hm].mean()
hm_data = hm_data.reindex(range(9)).T  # 행=피처, 열=T-0...T-8
hm_data.columns = [f'T-{i}' if i > 0 else 'T=0' for i in range(9)]
hm_data = hm_data[['T=0','T-1','T-2','T-3','T-4','T-5','T-6','T-7','T-8']]

fig, ax = plt.subplots(figsize=(16, 6))
sns.heatmap(
    hm_data, ax=ax,
    annot=True, fmt='.2f',
    cmap='RdYlGn_r',
    vmin=1, vmax=6,
    linewidths=0.8, linecolor='white',
    annot_kws={'size': 10, 'weight': 'bold'},
    yticklabels=labels_hm,
    cbar_kws={'label': '버킷값 (1=최상위 → 6=최하위)'}
)

# T=0 강조
ax.add_patch(plt.Rectangle((0, 0), 1, len(feats_hm),
             fill=False, edgecolor=C_CLOSED, linewidth=3, zorder=5))

ax.set_title('폐업 전 신호 강도 히트맵 (T-8 → T=0)\n'
             '붉을수록 해당 피처 하위권 (위험 신호)', fontsize=13, pad=15)
ax.set_xlabel('폐업까지 남은 개월', fontsize=12)
ax.tick_params(axis='y', rotation=0, labelsize=10)
ax.tick_params(axis='x', labelsize=11)

# 골든타임 화살표
ax.annotate(' 골든타임\n약 T-8~T-6',
            xy=(0.5, -0.15), xytext=(7.5, -0.15),
            xycoords=('axes fraction', 'axes fraction'),
            textcoords=('axes fraction', 'axes fraction'),
            fontsize=10, color=C_ACCENT,
            arrowprops=dict(arrowstyle='->', color=C_ACCENT, lw=1.5))

plt.tight_layout()
plt.savefig(f'{OUT_DIR}eda_10_signal_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print(" 저장: eda_10_signal_heatmap.png")


---
## 9. 요약 — EDA 주요 발견

| # | 발견 | 시사점 |
|---|---|---|
| 1 | **성수2가/왕십리** 상권은 매출은 높지만 폐업률도 높음 → 경쟁 심화 | 매출 수준만으로 안정 여부 판단 불가 |
| 2 | **마장동** 등 일부 상권은 매출 낮지만 폐업률 낮음 → 안정적 | 상권별 베이스라인 비교 필요성 |
| 3 | **골든타임 약 T-8개월**: 폐업 8개월 전부터 매출·재방문 버킷 하락 신호 | 조기 개입 가능 시점 확인 |
| 4 | **재방문율(MCT_UE_CLN_REU_RAT)** 효과 크기 가장 큼 | EWS 핵심 피처로 활용 |
| 5 | **업종 내 매출 순위** 폐업 점포에서 하위권 집중 | 절대 매출보다 상대적 위치가 더 강한 신호 |
| 6 | 피처 간 다중공선성 낮음 (Spearman r < 0.5) | 독립 신호로 가중 결합 가능 |

> **다음 단계**: 03_feature_engineering.ipynb — 추세 피처, 시간감쇠 스냅샷, 피어 그룹 순위 계산
